In [1]:
import pandas as pd
import numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler,LabelEncoder
from itertools import product
from IPython.display import display, clear_output
from skopt import gp_minimize
from skopt.space import Real, Integer, Categorical

In [2]:
df_train = pd.read_csv("train.csv", parse_dates=["date"])
df_test = pd.read_csv("test.csv", parse_dates=["date"])
df_stores = pd.read_csv("stores.csv")
df_oil = pd.read_csv("oil.csv", parse_dates=["date"])
df_holidays = pd.read_csv("holidays_events.csv", parse_dates=["date"])

In [3]:
# 檢查油價資料的缺漏值 #

# 設定日期為索引，方便後續處理
df_oil = df_oil.set_index('date')

# 我們需要涵蓋 df_train_mod (訓練集) 和 df_test (測試集) 的所有日期
start_date = df_train['date'].min()
end_date = df_test['date'].max()
# 日期範圍
full_date_range = pd.date_range(start=start_date, end=end_date, freq='D')
print(f"油價資料處理範圍: {start_date.date()} 到 {end_date.date()}")

# 重標索引
df_oil = df_oil.reindex(full_date_range)

# 索引名稱重新命名為 'date'
df_oil.index.name = 'date'

# 線性插值 (Interpolate)
# limit_direction='both' 確保頭尾的缺值也能被處理
df_oil['dcoilwtico'] = df_oil['dcoilwtico'].interpolate(method='linear', limit_direction='both')

# 重置索引
df_oil = df_oil.reset_index()

# 檢查是否有任何 NaN 殘留
print(f"缺漏值數量 (處理後): {df_oil['dcoilwtico'].isna().sum()}")
df_oil


油價資料處理範圍: 2013-01-01 到 2017-08-31
缺漏值數量 (處理後): 0


,date,dcoilwtico
0,2013-01-01,93.140000
1,2013-01-02,93.140000
2,2013-01-03,92.970000
3,2013-01-04,93.120000
4,2013-01-05,93.146667
...,...,...
1699,2017-08-27,46.816667
1700,2017-08-28,46.400000
1701,2017-08-29,46.460000
1702,2017-08-30,45.960000


In [4]:
# 清洗假日資料 #

# 紀錄被轉移的內容
condition_transferred = df_holidays['transferred'] == True

# 移除補班日 (type == 'Work Day')
# 這些日子雖然在表裡，但其實是要上班的
condition_workday = df_holidays['type'] == 'Work Day'

# 保留 "既沒有被轉移" 且 "不是補班日" 的資料
df_holidays_clean = df_holidays[~condition_transferred & ~condition_workday].copy()

# 需要保留日期、類型、地區資訊
df_holidays_clean = df_holidays_clean[['date', 'type', 'locale', 'locale_name', 'description']]
print(f"清洗後假日資料筆數: {df_holidays_clean.shape[0]}")
print("-" * 30)
print("清洗後範例 (前5筆):")
display(df_holidays_clean.head())

# 檢查是否還有 Work Day 或 transferred == True (理論上要是 0)
check_errors = df_holidays_clean[
    (df_holidays_clean['type'] == 'Work Day') |
    (df_holidays['transferred'] == True) 
]
print(check_errors)

清洗後假日資料筆數: 333
------------------------------
清洗後範例 (前5筆):


,date,type,locale,locale_name,description
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba


Empty DataFrame
Columns: [date, type, locale, locale_name, description]
Index: []


C:\Users\YoYo Chen\AppData\Local\Temp\ipykernel_27508\234136613.py:21: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  check_errors = df_holidays_clean[


In [5]:
# 合併訓練集與測試集，並利用part進行分辨
df_train['part'] = 'train'
df_test['part'] = 'test'

# 縱向合併 (訓練集 + 測試集)
df_main = pd.concat([df_train, df_test], ignore_index=True, sort=False)
print(f"合併後總筆數: {df_main.shape}")

# 建立日期特徵 (Date Features)
df_main['month'] = df_main['date'].dt.month
df_main['day'] = df_main['date'].dt.day
df_main['dayofweek'] = df_main['date'].dt.dayofweek  # 0=Monday, 6=Sunday

# 建立發薪日特徵 (Payday Logic) 每個月的 15 號 OR 每個月的最後一天 (is_month_end)，再轉成 0/1 標籤
df_main['is_payday'] = (df_main['day'] == 15) | (df_main['date'].dt.is_month_end)
df_main['is_payday'] = df_main['is_payday'].astype(int)
print("日期與發薪日特徵已建立。")

# 排序順序：先分店 -> 再商品 -> 最後按日期
df_main = df_main.sort_values(['store_nbr', 'family', 'date'])



print("-" * 30)
display(df_main[df_main['part'] == 'test'].head()[['date', 'store_nbr', 'family']])

合併後總筆數: (3029400, 7)
日期與發薪日特徵已建立。
------------------------------


,date,store_nbr,family
3000888,2017-08-16,1,AUTOMOTIVE
3002670,2017-08-17,1,AUTOMOTIVE
3004452,2017-08-18,1,AUTOMOTIVE
3006234,2017-08-19,1,AUTOMOTIVE
3008016,2017-08-20,1,AUTOMOTIVE


In [6]:
# 合併 Stores,Oil,Holidays
df_final = pd.merge(df_main, df_stores, on='store_nbr', how='left')
df_final = pd.merge(df_final, df_oil, on='date', how='left')
df_final = pd.merge(df_final, df_holidays_clean, on='date', how='left')

# 重新命名衝突的欄位
rename_dict = {
    'type_x': 'store_type',
    'type_y': 'holiday_type'
}
df_final = df_final.rename(columns=rename_dict)


#判斷假日的適用地區是否符合商店位置
is_national = df_final['locale'] == 'National'
is_regional = (df_final['locale'] == 'Regional') & (df_final['locale_name'] == df_final['state'])
is_local = (df_final['locale'] == 'Local') & (df_final['locale_name'] == df_final['city'])
# 只要符合其中一個，就是有效假日
mask_holiday_match = is_national | is_regional | is_local

# 準備要清除的欄位
cols_to_clear = ['holiday_type', 'locale', 'locale_name', 'description']
condition_to_clear = (~mask_holiday_match) & (df_final['holiday_type'].notna())
# 執行清除 (設為 NaN)
df_final.loc[condition_to_clear, cols_to_clear] = np.nan

# 6. 去除重複 (De-duplication)
df_final = df_final.drop_duplicates(subset=['date', 'store_nbr', 'family'])
print(f"合併前筆數: {df_main.shape[0]}")
print(f"合併後筆數: {df_final.shape[0]}")

# 移除 description 欄位
if 'description' in df_final.columns:
    df_final = df_final.drop(columns=['description'])
    print("已移除 description 欄位。")

# 油價補值 (後向填補)
df_final['dcoilwtico'] = df_final['dcoilwtico'].fillna(method='bfill')

# 假日相關補值
df_final['holiday_type'] = df_final['holiday_type'].fillna('WorkDay') # 沒放假就是工作日
df_final['locale'] = df_final['locale'].fillna('Normal')              # 沒有特定地區就是 Normal
# locale_name 也補 'Normal'
df_final['locale_name'] = df_final['locale_name'].fillna('Normal')

# 設定要編碼的欄位
cols_to_encode = ['family', 'city', 'state', 'store_type', 'holiday_type', 'locale', 'locale_name']

le = LabelEncoder()
for col in cols_to_encode:
    df_final[col] = df_final[col].astype(str)
    df_final[col] = le.fit_transform(df_final[col])
print("編碼完成。最終資料集準備就緒！")

# 檢查一下是否還有 NaN 
print("locale_name 缺值數:", df_final['locale_name'].isna().sum())
display(df_final.head())

合併前筆數: 3029400
合併後筆數: 3029400
已移除 description 欄位。


C:\Users\YoYo Chen\AppData\Local\Temp\ipykernel_27508\3462507588.py:38: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_final['dcoilwtico'] = df_final['dcoilwtico'].fillna(method='bfill')


編碼完成。最終資料集準備就緒！
locale_name 缺值數: 0


,id,date,store_nbr,family,sales,onpromotion,part,month,day,dayofweek,is_payday,city,state,store_type,cluster,dcoilwtico,holiday_type,locale,locale_name
0,0,2013-01-01,1,0,0.0,0,train,1,1,1,0,18,12,3,13,93.140000,3,1,4
1,1782,2013-01-02,1,0,2.0,0,train,1,2,2,0,18,12,3,13,93.140000,5,2,16
2,3564,2013-01-03,1,0,3.0,0,train,1,3,3,0,18,12,3,13,92.970000,5,2,16
3,5346,2013-01-04,1,0,3.0,0,train,1,4,4,0,18,12,3,13,93.120000,5,2,16
4,7128,2013-01-05,1,0,5.0,0,train,1,5,5,0,18,12,3,13,93.146667,5,2,16


In [7]:
val_start_date = '2017-04-15'
val_end_date = '2017-08-15'

# Rebuild model_df with lag features and refreshed splits/scaling
base_cols = [
    'id','date','part','dcoilwtico','locale_name','store_type','city',
    'dayofweek','sales','store_nbr','onpromotion','locale','holiday_type',
    'state','family'
]
model_df = df_final[base_cols].copy()
model_df['year'] = model_df['date'].dt.year
model_df['month'] = model_df['date'].dt.month
model_df['day'] = model_df['date'].dt.day

# Train/val/test split
train_df = model_df[model_df['part'] == 'train'].sort_values(['store_nbr','family','date']).reset_index(drop=True)
test_df = model_df[model_df['part'] == 'test'].sort_values(['store_nbr','family','date']).reset_index(drop=True)

val_mask = (train_df['date'] >= val_start_date) & (train_df['date'] <= val_end_date)
val_df = train_df.loc[val_mask].copy()
train_df = train_df.loc[~val_mask].copy()

# Keep 'part' so we can filter test rows when building sequences
# (drop later if needed after sequence construction)
# for df_ in [train_df, val_df, test_df]:
#     if 'part' in df_.columns:
#         df_.drop(columns=['part'], inplace=True)

CATEGORICAL_COLS = [
    'locale_name', 'store_type', 'city', 'dayofweek',
    'store_nbr', 'locale', 'holiday_type', 'state',
    'family', 'year', 'month', 'day'
]

for col in CATEGORICAL_COLS:
    # 用 train_df 的 mapping 來轉換三個 dataframe
    mapping = {v: i for i, v in enumerate(sorted(train_df[col].unique()))}
    
    train_df[col] = train_df[col].map(mapping).astype(int)
    val_df[col]   = val_df[col].map(mapping).fillna(-1).astype(int)
    test_df[col]  = test_df[col].map(mapping).fillna(-1).astype(int)




In [8]:
NUMERIC_COLS = [
    'dcoilwtico','onpromotion',
]
scale_cols = NUMERIC_COLS
scaler = MinMaxScaler()
scaler.fit(train_df[scale_cols])
for df_ in [train_df, val_df, test_df]:
    df_.loc[:, scale_cols] = scaler.transform(df_[scale_cols])

feature_cols = NUMERIC_COLS + CATEGORICAL_COLS


print(f"Feature columns ({len(feature_cols)}): {feature_cols}")
print(f"train rows: {train_df.shape[0]}, val rows: {val_df.shape[0]}, test rows: {test_df.shape[0]}")
print("TRAIN null count:", train_df.isna().sum().sum())
print("VAL null count:", val_df.isna().sum().sum())
print("TEST null count:", test_df.isna().sum().sum())
print("train_df year unique:", sorted(train_df['year'].unique()))
print("val_df   year unique:", sorted(val_df['year'].unique()))


display(train_df.head())


Feature columns (14): ['dcoilwtico', 'onpromotion', 'locale_name', 'store_type', 'city', 'dayofweek', 'store_nbr', 'locale', 'holiday_type', 'state', 'family', 'year', 'month', 'day']
train rows: 2781702, val rows: 219186, test rows: 28512
TRAIN null count: 0
VAL null count: 0
TEST null count: 28512
train_df year unique: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
val_df   year unique: [np.int64(4)]


C:\Users\YoYo Chen\AppData\Local\Temp\ipykernel_27508\869314914.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_.loc[:, scale_cols] = scaler.transform(df_[scale_cols])
C:\Users\YoYo Chen\AppData\Local\Temp\ipykernel_27508\869314914.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_.loc[:, scale_cols] = scaler.transform(df_[scale_cols])
C:\Users\YoYo Chen\AppData\Local\Temp\ipykernel_27508\869314914.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a co

,id,date,part,dcoilwtico,locale_name,store_type,city,dayofweek,sales,store_nbr,onpromotion,locale,holiday_type,state,family,year,month,day
0,0,2013-01-01,train,0.792965,4,3,18,1,0.0,0,0.0,1,3,12,0,0,0,0
1,1782,2013-01-02,train,0.792965,16,3,18,2,2.0,0,0.0,2,5,12,0,0,0,1
2,3564,2013-01-03,train,0.790951,16,3,18,3,3.0,0,0.0,2,5,12,0,0,0,2
3,5346,2013-01-04,train,0.792728,16,3,18,4,3.0,0,0.0,2,5,12,0,0,0,3
4,7128,2013-01-05,train,0.793044,16,3,18,5,5.0,0,0.0,2,5,12,0,0,0,4


In [9]:
# Sequence builders
def build_sequences(df, feature_cols, target_col, lookback):
    X_list, y_list = [], []
    for (_, _), g in df.groupby(['store_nbr', 'family'], sort=False):
        g_sorted = g.sort_values('date')
        arr = g_sorted[feature_cols + [target_col]].to_numpy()
        for i in range(len(g_sorted) - lookback):
            window = arr[i:i+lookback, :-1]
            target = arr[i+lookback, -1]
            if np.isnan(target) or np.any(np.isnan(window)):
                continue
            X_list.append(window)
            y_list.append(target)
    if not X_list:
        return np.empty((0, lookback, len(feature_cols))), np.array([])
    return np.stack(X_list), np.array(y_list)

def build_sequences_for_predict(df, feature_cols, lookback):
    X_list, id_list = [], []
    for (_, _), g in df.groupby(['store_nbr', 'family'], sort=False):
        g_sorted = g.sort_values('date')
        feats = g_sorted[feature_cols].to_numpy()
        ids = g_sorted['id'].to_numpy()
        parts = g_sorted['part'].astype(str).to_numpy() if 'part' in g_sorted.columns else None
        for i in range(len(g_sorted) - lookback):
            target_idx = i + lookback
            if parts is not None and parts[target_idx] != 'test':
                continue
            window = feats[i:target_idx]
            if np.any(np.isnan(window)):
                continue
            X_list.append(window)
            id_list.append(ids[target_idx])
    if not X_list:
        return np.empty((0, lookback, len(feature_cols))), np.array([])
    return np.stack(X_list), np.array(id_list)


In [10]:

# Build sequences
lookback = 45
X_train, y_train = build_sequences(train_df, feature_cols, 'sales', lookback)
X_val,   y_val   = build_sequences(val_df,   feature_cols, 'sales', lookback)

# 👉 先存一份「原始 y」，之後算 RMSLE / 最後訓練會用到
y_train_orig = y_train.copy()
y_val_orig   = y_val.copy()

# 👉 模型訓練時使用 log1p 後的 y
y_train = np.log1p(y_train)
y_val   = np.log1p(y_val)

# Use all available history for test sequences so each test row has a lookback window
all_history = pd.concat([train_df, val_df, test_df], ignore_index=True, sort=False)
all_history = all_history.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)
X_test, test_ids = build_sequences_for_predict(all_history, feature_cols, lookback)

# Drop part now that sequence windows are built
for df_ in [train_df, val_df, test_df]:
    if 'part' in df_.columns:
        df_.drop(columns=['part'], inplace=True)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print("y_train (log) describe:")
print(pd.Series(y_train).describe())

print("\ny_train_orig (raw) describe:")
print(pd.Series(y_train_orig).describe())

print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, test_ids: {test_ids.shape}")



X_train: (2701512, 45, 14), y_train: (2701512,)
y_train (log) describe:
count    2.701512e+06
mean     2.895797e+00
std      2.699602e+00
min      0.000000e+00
25%      0.000000e+00
50%      2.397895e+00
75%      5.267858e+00
max      1.173381e+01
dtype: float64

y_train_orig (raw) describe:
count    2.701512e+06
mean     3.527131e+02
std      1.090623e+03
min      0.000000e+00
25%      0.000000e+00
50%      1.000000e+01
75%      1.930000e+02
max      1.247170e+05
dtype: float64
X_val: (138996, 45, 14), y_val: (138996,)
X_test: (28512, 45, 14), test_ids: (28512,)


In [11]:
# ?????? cardinality???????????

cat_cardinalities = {}
for col in CATEGORICAL_COLS:
    cat_cardinalities[col] = train_df[col].nunique() + 1


cat_cardinalities


{'locale_name': 25,
 'store_type': 6,
 'city': 23,
 'dayofweek': 8,
 'store_nbr': 55,
 'locale': 5,
 'holiday_type': 7,
 'state': 17,
 'family': 34,
 'year': 6,
 'month': 13,
 'day': 32}

In [12]:
def build_model_with_embeddings(
    lookback,
    feature_cols,
    numeric_cols,
    categorical_cols,
    cat_cardinalities,
    rnn_type='gru',
    units=128,
    dropout=0.1,
    lr=1e-3,
    bidirectional=False
):
    """
    建立 Embedding + RNN 模型，可選 rnn/lstm/gru
    """
    n_features = len(feature_cols)

    inp = tf.keras.Input(shape=(lookback, n_features), name='seq_input')

    # 取 numeric
    num_indices = [feature_cols.index(c) for c in numeric_cols]
    num_tensor = tf.keras.layers.Lambda(
        lambda t: tf.gather(t, num_indices, axis=-1),
        name='numeric_slice'
    )(inp)

    # categorical embedding
    emb_outputs = []
    for col in categorical_cols:
        idx = feature_cols.index(col)

        col_tensor = tf.keras.layers.Lambda(
            lambda t, i=idx: t[:, :, i],
            name=f'slice_{col}'
        )(inp)
        col_tensor = tf.keras.layers.Lambda(lambda t: tf.cast(t, tf.int32))(col_tensor)

        vocab_size = cat_cardinalities[col]
        emb_dim = min(16, vocab_size // 2 + 1)

        emb = tf.keras.layers.Embedding(
            input_dim=vocab_size,
            output_dim=emb_dim,
            name=f'emb_{col}'
        )(col_tensor)

        emb_outputs.append(emb)

    seq_repr = tf.keras.layers.Concatenate(axis=-1)([num_tensor] + emb_outputs)

    layer_map = {
        'rnn' : tf.keras.layers.SimpleRNN,
        'lstm': tf.keras.layers.LSTM,
        'gru' : tf.keras.layers.GRU
    }
    rnn_cls = layer_map[rnn_type]

    core = rnn_cls(units, return_sequences=False, dropout=dropout)
    if bidirectional:
        core = tf.keras.layers.Bidirectional(core)

    h = core(seq_repr)
    out = tf.keras.layers.Dense(1)(h)

    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='mse'
    )
    return model


In [13]:
# RMSLE metric, model builder, and Bayesian optimizer

def rmsle(y_true, y_pred):
    y_true = np.clip(y_true, 0, None)
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean(np.square(np.log1p(y_pred) - np.log1p(y_true))))

def build_model(kind, input_shape, units=64, dropout=0.0, lr=1e-3, bidirectional=False):
    layer_map = {
        'rnn': tf.keras.layers.SimpleRNN,
        'lstm': tf.keras.layers.LSTM,
        'gru': tf.keras.layers.GRU
    }
    '''
    layer_map = {
        'rnn': tf.keras.layers.SimpleRNN,
        'lstm': tf.keras.layers.LSTM,
        'gru': tf.keras.layers.GRU
    }
    '''
    rnn_layer = layer_map[kind]

    core = rnn_layer(units, return_sequences=False, dropout=dropout, recurrent_dropout=0.1)
    if bidirectional:
        core = tf.keras.layers.Bidirectional(core)

    model = tf.keras.Sequential([
    core,
    tf.keras.layers.Dropout(dropout),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)   # 這裡可以不用 relu，輸出 log1p(sales) 就好
])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr, clipnorm=1.0),  # clip 梯度
        loss='msle'  # 直接最小化 log1p 對應的 msle
    )
    return model


def run_embedding_bayes_for_model(
    model_type,             # 'rnn' / 'lstm' / 'gru'
    X_train, y_train,
    X_val, y_val,
    feature_cols,
    numeric_cols,
    categorical_cols,
    cat_cardinalities,
    lookback,
    n_calls=30,             # 這一個模型要試幾組超參數
    n_initial_points=10,    # 前幾組用隨機探索
    random_state=42
):
    """
    對「單一模型類型」（model_type=fixed）做 Bayesian Optimization。
    例如：只對 LSTM 搜尋 units/dropout/lr/... 的最佳組合。

    回傳：
      - res_df: 每次嘗試的超參數與 RMSLE（已排序）
      - bo_result: skopt 的原始結果物件（可選）
    """

    # 定義這個模型要搜尋的超參數空間（不含 rnn_type，因為已固定）
    search_space = [
    # units: 64 ~ 180, 間隔 16（64, 80, 96, ... 176）
    Integer(64, 180, name='units'),

    # dropout: 建議 0.15 ~ 0.35
    Real(0.15, 0.35, name='dropout'),

    # batch_size: 32 / 64 / 128
    Categorical([32, 64, 128], name='batch_size'),

    # epochs: 40 ~ 120
    Integer(40, 120, name='epochs'),

    # lr: 用 log-uniform 分佈，約 2e-4 ~ 1.5e-3（對應我給的 0.0002 ~ 0.0015）
    Real(2e-4, 1.5e-3, prior='log-uniform', name='lr'),

    # bidirectional: True / False
    Categorical([False, True], name='bidirectional'),
]


    results = []

    def objective(params):
        units, dropout, batch_size, epochs, lr, bidirectional = params

        cfg = {
            'units': int(units),
            'dropout': float(dropout),
            'batch_size': int(batch_size),
            'epochs': int(epochs),
            'lr': float(lr),
            'bidirectional': bool(bidirectional)
        }

        print(f"\n[BO-{model_type.upper()}] Training config = {cfg}")

        # 這裡使用你前面定義好的 build_model_with_embeddings
        model = build_model_with_embeddings(
            lookback=lookback,
            feature_cols=feature_cols,
            numeric_cols=numeric_cols,
            categorical_cols=categorical_cols,
            cat_cardinalities=cat_cardinalities,
            rnn_type=model_type,           # 🔒 固定為這一次的模型類型
            units=cfg['units'],
            dropout=cfg['dropout'],
            lr=cfg['lr'],
            bidirectional=cfg['bidirectional']
        )

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                patience=3,
                monitor='val_loss',
                restore_best_weights=True
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=2,
                min_lr=1e-5,
                verbose=1
            )
        ]


        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=cfg['epochs'],
            batch_size=cfg['batch_size'],
            verbose=0,
            callbacks=callbacks
        )

        # 模型輸出是 log1p(sales)，轉回原尺度再算 RMSLE
        preds_log = model.predict(X_val, verbose=0).squeeze()
        preds = np.expm1(preds_log)

        # 這裡用你前面準備好的 y_val_orig（原始 sales）
        score = rmsle(y_val_orig, preds)

        result_row = {
            'model_type': model_type,
            **cfg,
            'rmsle': float(score)
        }
        results.append(result_row)

        print(f"[BO-{model_type.upper()}] → RMSLE = {score:.4f}")

        # gp_minimize 是越小越好 → RMSLE 本來就是越小越好
        return score

    # 啟動 Bayesian Optimization
    bo_result = gp_minimize(
        func=objective,
        dimensions=search_space,
        n_calls=n_calls,
        n_initial_points=n_initial_points,
        random_state=random_state,
        verbose=False
    )

    res_df = pd.DataFrame(results).sort_values('rmsle')
    return res_df, bo_result

In [14]:
# === 這一格負責實際執行：對 RNN / LSTM / GRU 各跑一輪 BO，最後比較 ===

# Bayesian Optimization 總嘗試次數（每種模型各跑 n_calls 次）
n_calls = 25          # 原本 10 → 25
n_initial_points = 15 # 原本 10 → 15


# 限制訓練樣本數（加速用，可依機器算力調整或拿掉）
max_train_samples = 200000
if len(X_train) > max_train_samples:
    X_train_use = X_train[:max_train_samples]
    y_train_use = y_train[:max_train_samples]
else:
    X_train_use, y_train_use = X_train, y_train

model_types = ['rnn', 'lstm', 'gru'] 

all_results = []   # 存每個模型所有嘗試的結果
summary_rows = []  # 存每個模型「最佳一組」的表

for m in model_types:
    print("\n" + "=" * 60)
    print(f"🔍 Start Bayesian Optimization for model_type = {m.upper()}")
    print("=" * 60)

    res_df, bo_res = run_embedding_bayes_for_model(
        model_type=m,
        X_train=X_train_use,
        y_train=y_train_use,
        X_val=X_val,
        y_val=y_val,
        feature_cols=feature_cols,
        numeric_cols=NUMERIC_COLS,
        categorical_cols=CATEGORICAL_COLS,
        cat_cardinalities=cat_cardinalities,
        lookback=lookback,
        n_calls=n_calls,
        n_initial_points=n_initial_points,
        random_state=42
    )

    all_results.append(res_df)

    # 取出這個模型類型中 RMSLE 最小的一組
    best_row = res_df.iloc[0]
    summary_rows.append({
        'model_type': m,
        'best_rmsle': best_row['rmsle'],
        'best_units': best_row['units'],
        'best_dropout': best_row['dropout'],
        'best_batch_size': best_row['batch_size'],
        'best_epochs': best_row['epochs'],
        'best_lr': best_row['lr'],
        'best_bidirectional': best_row['bidirectional']
    })

# 把所有嘗試過的結果合併在一起（可選，方便之後分析）
results_all = pd.concat(all_results, ignore_index=True).sort_values('rmsle')
print("\n🔥 Top 10 configs across ALL models:")
display(results_all.head(10))

# 每種類型的最佳結果 summary
summary_df = pd.DataFrame(summary_rows).sort_values('best_rmsle')
print("\n🏆 Best result for each model_type:")
display(summary_df)

# 找出「全場最強」的那一個模型 + 超參數
best_overall = summary_df.iloc[0]
print("\n🥇 Overall best model_type & config:")
print(best_overall)

# 轉成你後續會使用的 best_cfg 結構（方便直接塞回 build_model 重訓）
best_cfg = {
    'rnn_type': best_overall['model_type'],
    'units': int(best_overall['best_units']),
    'dropout': float(best_overall['best_dropout']),
    'batch_size': int(best_overall['best_batch_size']),
    'epochs': int(best_overall['best_epochs']),
    'lr': float(best_overall['best_lr']),
    'bidirectional': bool(best_overall['best_bidirectional'])
}
print("\n✅ best_cfg (for retraining on full data) =", best_cfg)



🔍 Start Bayesian Optimization for model_type = RNN

[BO-RNN] Training config = {'units': 156, 'dropout': 0.18668695797323276, 'batch_size': 128, 'epochs': 88, 'lr': 0.0004910898607972045, 'bidirectional': False}


Epoch 3: ReduceLROnPlateau reducing learning rate to 0.0002455449430271983.
[BO-RNN] → RMSLE = 0.9170

[BO-RNN] Training config = {'units': 117, 'dropout': 0.21674172222780436, 'batch_size': 32, 'epochs': 92, 'lr': 0.00022407509193348162, 'bidirectional': True}

Epoch 5: ReduceLROnPlateau reducing learning rate to 0.00011203754547750577.
[BO-RNN] → RMSLE = 0.9497

[BO-RNN] Training config = {'units': 173, 'dropout': 0.15015575316820287, 'batch_size': 128, 'epochs': 89, 'lr': 0.0006859050226136135, 'bidirectional': False}

Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0003429525240790099.
[BO-RNN] → RMSLE = 0.8840

[BO-RNN] Training config = {'units': 67, 'dropout': 0.25495493205167785, 'batch_size': 64, 'epochs': 44, 'lr': 0.0014227406173477584, 'bidirectional': Fal

,model_type,units,dropout,batch_size,epochs,lr,bidirectional,rmsle
0,rnn,173,0.150156,128,89,0.000686,False,0.884035
1,rnn,180,0.150000,64,72,0.000704,False,0.889761
2,rnn,155,0.152352,64,59,0.001303,True,0.899295
25,lstm,130,0.156263,128,76,0.000443,True,0.904590
3,rnn,173,0.150000,64,59,0.001197,False,0.904640
50,gru,91,0.150000,128,40,0.001500,True,0.908986
4,rnn,140,0.332992,128,76,0.000242,False,0.909511
51,gru,83,0.150000,128,60,0.001500,True,0.910595
26,lstm,140,0.332992,128,76,0.000242,False,0.912572
52,gru,74,0.158804,128,40,0.000200,False,0.912673



🏆 Best result for each model_type:


,model_type,best_rmsle,best_units,best_dropout,best_batch_size,best_epochs,best_lr,best_bidirectional
0,rnn,0.884035,173,0.150156,128,89,0.000686,False
1,lstm,0.904590,130,0.156263,128,76,0.000443,True
2,gru,0.908986,91,0.150000,128,40,0.001500,True



🥇 Overall best model_type & config:
model_type                 rnn
best_rmsle            0.884035
best_units                 173
best_dropout          0.150156
best_batch_size            128
best_epochs                 89
best_lr               0.000686
best_bidirectional       False
Name: 0, dtype: object

✅ best_cfg (for retraining on full data) = {'rnn_type': 'rnn', 'units': 173, 'dropout': 0.15015575316820287, 'batch_size': 128, 'epochs': 89, 'lr': 0.0006859050226136135, 'bidirectional': False}


In [15]:
# ? log1p(y) ???????
X_full = np.concatenate([X_train, X_val], axis=0)
y_full_log = np.concatenate([y_train, y_val], axis=0)  # ??? log1p ? y

best_model = build_model_with_embeddings(
    lookback=lookback,
    feature_cols=feature_cols,
    numeric_cols=NUMERIC_COLS,
    categorical_cols=CATEGORICAL_COLS,
    cat_cardinalities=cat_cardinalities,
    rnn_type=best_cfg['rnn_type'],
    units=int(best_cfg['units']),
    dropout=float(best_cfg['dropout']),
    lr=float(best_cfg['lr']),
    bidirectional=bool(best_cfg['bidirectional'])
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=5,
        monitor='val_loss',
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        patience=3,
        factor=0.5,
        min_lr=1e-5
    )
]

best_model.fit(
    X_full, y_full_log,
    epochs=int(best_cfg['epochs']),
    batch_size=int(best_cfg['batch_size']),
    verbose=1,
    callbacks=callbacks,
    validation_split=0.1
)

# ? test ???? log1p(sales) ??? expm1
test_preds_log = best_model.predict(X_test, verbose=0).squeeze()
test_preds = np.expm1(test_preds_log)

submission = pd.DataFrame({
    'id': test_ids.astype(int),
    'sales': np.clip(test_preds, 0, None)
})
submission_path = 'submission_rnn_lstm_gru.csv'
submission.to_csv(submission_path, index=False)
print(f"Saved submission to {submission_path} | rows: {len(submission)} | sales>=0: {(submission['sales']>=0).mean():.3f}")


Epoch 1/89
19973/19973 ━━━━━━━━━━━━━━━━━━━━ 507s 25ms/step - loss: 0.7856 - val_loss: 1.2638 - learning_rate: 6.8591e-04
Epoch 2/89
19973/19973 ━━━━━━━━━━━━━━━━━━━━ 496s 25ms/step - loss: 0.4718 - val_loss: 1.4479 - learning_rate: 6.8591e-04
Epoch 3/89
19973/19973 ━━━━━━━━━━━━━━━━━━━━ 500s 25ms/step - loss: 0.4091 - val_loss: 1.5750 - learning_rate: 6.8591e-04
Epoch 4/89
19973/19973 ━━━━━━━━━━━━━━━━━━━━ 501s 25ms/step - loss: 0.4199 - val_loss: 1.4876 - learning_rate: 6.8591e-04
Epoch 5/89
19973/19973 ━━━━━━━━━━━━━━━━━━━━ 497s 25ms/step - loss: 0.3452 - val_loss: 1.5639 - learning_rate: 3.4295e-04
Epoch 6/89
19973/19973 ━━━━━━━━━━━━━━━━━━━━ 495s 25ms/step - loss: 0.3215 - val_loss: 1.6495 - learning_rate: 3.4295e-04
Saved submission to submission_rnn_lstm_gru.csv | rows: 28512 | sales>=0: 1.000
